# Makemore MLP — E01 기준 모델 재구성

다른 컴퓨터에서 학습·평가 구현, 학습률을 0.01로 낮춘 뒤 약 20,000회 학습까지 진행했다는 학습자의 설명을 바탕으로, 명시적 예외 요청에 따라 AI가 실행 준비 코드를 재구성했다.

당시 코드·출력·가중치의 복구본은 아니다. 배치 32와 E02 초기화는 현재 보관본을 기반으로 한 재구성 설정이다. 학습률을 낮추기 전의 가중치와 학습 횟수는 확인되지 않았다. 따라서 여기서는 새 초기화에서 학습률 0.01로 20,000회 실행하도록 준비했다. 당시의 이어 학습과는 출발 가중치가 다르다. 아래 실행 결과는 이 컴퓨터의 새 실험이다.

프로젝트 `.venv` 커널에서 위에서부터 실행한다. 초기화 셀을 다시 실행하면 모델과 학습 기록이 초기화된다. 학습 셀을 다시 실행하면 현재 모델에서 추가 학습한다. test는 튜닝에 사용하지 않는다.

참고: [공식 구현](https://github.com/karpathy/nn-zero-to-hero/blob/master/lectures/makemore/makemore_part2_mlp.ipynb), `practice/deep-learning/makemore-mlp-e02-initialization-training.ipynb`.


## CUDA 학습률 비교

동일한 CUDA 환경에서 모델과 배치 생성기의 seed를 초기화하고, 각 학습률로 20,000회씩 학습했다.

- 모델 seed: 2147483647, 배치 seed: 42, 데이터 분할 seed: 42
- 문맥 길이 3, 임베딩 차원 10, 은닉층 크기 200, 배치 크기 32
- 두 실행의 초기 손실: train 3.3480371374486655, dev 3.3487788334873647

| 학습률 | 학습 횟수 | train loss | dev loss |
|---|---:|---:|---:|
| 0.1 | 20,000 | 2.2630279827600956 | 2.296643974322721 |
| 0.01 | 20,000 | 2.2722113385523612 | 2.2844026739406313 |

0.1 결과는 이 대화에서 앞서 확인한 저장 출력에서 옮겼다. 당시 평가 이름에는 0.01이 남아 있었지만 실제 코드와 학습 로그는 0.1이었다. 0.01 결과는 현재 학습 셀의 저장 출력에서 확인했다.

이번 한 seed의 비교에서는 0.1의 train 손실이 더 낮고, 0.01의 dev 손실이 더 낮았다. 모든 초기화에서 같은 우열이 성립한다고 단정하지 않는다. CPU에서 수행한 이전 실험은 이 표에 포함하지 않는다.


In [1]:
import random
from urllib.request import urlopen

import torch
import torch.nn.functional as F

with urlopen("https://raw.githubusercontent.com/karpathy/makemore/master/names.txt", timeout=30) as response:
    words = response.read().decode("utf-8").splitlines()

chars = sorted(set("".join(words)))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)
block_size = 3
embedding_dim = 10
hidden_size = 200
batch_size = 32
device = torch.device("cuda")
print("words:", len(words), "vocabulary:", vocab_size, "device:", device)

words: 32033 vocabulary: 27 device: cuda


In [2]:
def build_dataset(word_list):
    X, Y = [], []
    for word in word_list:
        context = [0] * block_size
        for ch in word + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return (
        torch.tensor(X, dtype=torch.long, device=device),
        torch.tensor(Y, dtype=torch.long, device=device),
    )

shuffled_words = words.copy()
random.Random(42).shuffle(shuffled_words)
n = len(shuffled_words)
train_end, dev_end = n * 8 // 10, n * 9 // 10
Xtr, Ytr = build_dataset(shuffled_words[:train_end])
Xdev, Ydev = build_dataset(shuffled_words[train_end:dev_end])
Xte, Yte = build_dataset(shuffled_words[dev_end:])
for name, X, Y in [("train", Xtr, Ytr), ("dev", Xdev, Ydev), ("test", Xte, Yte)]:
    print(name, X.shape, Y.shape)

train torch.Size([182625, 3]) torch.Size([182625])
dev torch.Size([22655, 3]) torch.Size([22655])
test torch.Size([22866, 3]) torch.Size([22866])


In [3]:
# 다시 실행하면 모델·난수 시퀀스·기록을 초기화합니다.
g_model = torch.Generator(device=device).manual_seed(2147483647)
g_batch = torch.Generator(device=device).manual_seed(42)
C = torch.randn((vocab_size, embedding_dim), generator=g_model, device=device, dtype=torch.float32)
W1 = torch.randn((block_size * embedding_dim, hidden_size), generator=g_model, device=device, dtype=torch.float32)
b1 = torch.randn(hidden_size, generator=g_model, device=device, dtype=torch.float32)
W2 = torch.randn((hidden_size, vocab_size), generator=g_model, device=device, dtype=torch.float32) * 0.02
b2 = torch.zeros(vocab_size, device=device, dtype=torch.float32)
parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad_(True)

steps_completed = 0
loss_history = []
evaluations = []
print("parameters:", sum(p.numel() for p in parameters))

parameters: 11897


In [4]:
def forward(X):
    emb = C[X]
    flat = emb.flatten(start_dim=1)
    h = torch.tanh(flat @ W1 + b1)
    return h @ W2 + b2

@torch.no_grad()
def evaluate(X, Y, chunk_size=4096):
    total_loss = 0.0
    for start in range(0, len(Y), chunk_size):
        logits = forward(X[start:start + chunk_size])
        total_loss += F.cross_entropy(
            logits, Y[start:start + chunk_size], reduction="sum"
        ).item()
    return total_loss / len(Y)

def record_evaluation(label):
    result = {
        "label": label,
        "step": steps_completed,
        "train_loss": evaluate(Xtr, Ytr),
        "dev_loss": evaluate(Xdev, Ydev),
    }
    evaluations.append(result)
    print(result)

def train_steps(num_steps, learning_rate, log_every=1000):
    global steps_completed
    for _ in range(num_steps):
        indices = torch.randint(len(Xtr), (batch_size,), generator=g_batch, device=device)
        loss = F.cross_entropy(forward(Xtr[indices]), Ytr[indices])
        for p in parameters:
            p.grad = None
        loss.backward()
        with torch.no_grad():
            for p in parameters:
                p -= learning_rate * p.grad
        steps_completed += 1
        loss_history.append((steps_completed, learning_rate, loss.item()))
        if steps_completed % log_every == 0:
            print(f"step={steps_completed}, lr={learning_rate}, batch loss (before update)={loss.item():.4f}")

record_evaluation("initial")

{'label': 'initial', 'step': 0, 'train_loss': 3.3480371374486655, 'dev_loss': 3.3487788334873647}


In [5]:
# 당시 lr 변경 전 모델은 없으므로, 여기서는 새 초기화부터 실행합니다.
# 재실행하면 현재 모델에서 20,000회 추가 학습합니다.
lr = 0.03
train_steps(20_000, learning_rate=lr)
record_evaluation("after 20,000 steps at lr=0.01")

step=1000, lr=0.03, batch loss (before update)=2.2919
step=2000, lr=0.03, batch loss (before update)=2.2493
step=3000, lr=0.03, batch loss (before update)=2.1640
step=4000, lr=0.03, batch loss (before update)=2.1975
step=5000, lr=0.03, batch loss (before update)=2.5422
step=6000, lr=0.03, batch loss (before update)=2.0844
step=7000, lr=0.03, batch loss (before update)=1.9588
step=8000, lr=0.03, batch loss (before update)=2.3658
step=9000, lr=0.03, batch loss (before update)=2.4353
step=10000, lr=0.03, batch loss (before update)=2.0640
step=11000, lr=0.03, batch loss (before update)=1.9425
step=12000, lr=0.03, batch loss (before update)=2.1789
step=13000, lr=0.03, batch loss (before update)=2.3962
step=14000, lr=0.03, batch loss (before update)=2.2133
step=15000, lr=0.03, batch loss (before update)=1.9555
step=16000, lr=0.03, batch loss (before update)=1.9777
step=17000, lr=0.03, batch loss (before update)=2.1126
step=18000, lr=0.03, batch loss (before update)=1.9862
step=19000, lr=0.03

In [6]:
print("learning rate:", lr, "completed steps:", steps_completed)

learning rate: 0.03 completed steps: 20000


## 기준 결과 확인 후 다음 설정 비교

학습자는 다른 컴퓨터에서 학습률을 0.01로 낮춘 뒤 약 20,000회 학습했다고 설명했다. 위 셀은 그 학습률과 횟수를 반영하지만 당시의 학습된 가중치에서 이어지는 실행은 아니다.

아래 추가 학습은 기본값 0으로 두었다. 기준 train/dev 결과를 확인한 뒤 다음 비교 설정을 결정한다. 이전 에이전트가 제안한 변경 항목은 현재 기록에 없어 특정 항목으로 단정하지 않는다. 추가 학습이 필요할 때만 `additional_steps`를 지정한다.

In [9]:
lr=0.01

additional_steps = 20000
if additional_steps > 0:
    train_steps(additional_steps, learning_rate=lr)
    record_evaluation("after lr=0.01 continuation")
else:
    print("추가 학습 대기: additional_steps를 지정하세요.")

step=41000, lr=0.01, batch loss (before update)=2.1233
step=42000, lr=0.01, batch loss (before update)=1.7753
step=43000, lr=0.01, batch loss (before update)=2.1077
step=44000, lr=0.01, batch loss (before update)=2.0241
step=45000, lr=0.01, batch loss (before update)=2.3634
step=46000, lr=0.01, batch loss (before update)=1.9408
step=47000, lr=0.01, batch loss (before update)=2.5523
step=48000, lr=0.01, batch loss (before update)=2.3283
step=49000, lr=0.01, batch loss (before update)=2.0471
step=50000, lr=0.01, batch loss (before update)=1.9154
step=51000, lr=0.01, batch loss (before update)=2.3714
step=52000, lr=0.01, batch loss (before update)=1.9582
step=53000, lr=0.01, batch loss (before update)=2.1487
step=54000, lr=0.01, batch loss (before update)=1.9509
step=55000, lr=0.01, batch loss (before update)=1.9931
step=56000, lr=0.01, batch loss (before update)=2.0843
step=57000, lr=0.01, batch loss (before update)=1.9629
step=58000, lr=0.01, batch loss (before update)=1.8924
step=59000

In [10]:
# 이번 커널에서 실제 평가한 결과만 표시합니다.
for result in evaluations:
    print(result)

{'label': 'initial', 'step': 0, 'train_loss': 3.3480371374486655, 'dev_loss': 3.3487788334873647}
{'label': 'after 20,000 steps at lr=0.01', 'step': 20000, 'train_loss': 2.231744363877481, 'dev_loss': 2.251827299092364}
{'label': 'after lr=0.01 continuation', 'step': 40000, 'train_loss': 2.1906464375855577, 'dev_loss': 2.216848084721916}
{'label': 'after lr=0.01 continuation', 'step': 60000, 'train_loss': 2.162177297227926, 'dev_loss': 2.190618965184286}
